In [ ]:
# ============================================================
# THRESHOLD SENSITIVITY ANALYSIS (per-class, one-vs-rest)
# Computes Sensitivity / Specificity / Accuracy across all thresholds,
# finds t_acc (accuracy-optimal) and t_bal (sensitivity=specificity),
# then re-evaluates the model using calibrated per-class thresholds
# to fix the sensitivity drop seen at the default t=0.50 cutoff.
# Session-resumable: safe against Kaggle kernel resets (loads y_true/
# y_prob/class_names from checkpoint if the kernel restarted).
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CKPT_YTRUE = os.path.join(OUTPUT_DIR, "threshold_y_true.npy")
CKPT_YPROB = os.path.join(OUTPUT_DIR, "threshold_y_prob.npy")
CKPT_CLASSNAMES = os.path.join(OUTPUT_DIR, "threshold_class_names.npy")

# ---- safety: reload from checkpoint if kernel was restarted ----
try:
    y_true
    y_prob
    class_names
except NameError:
    y_true = np.load(CKPT_YTRUE)
    y_prob = np.load(CKPT_YPROB)
    class_names = list(np.load(CKPT_CLASSNAMES, allow_pickle=True))
    print("Reloaded y_true / y_prob / class_names from checkpoint")

# ---- always save checkpoint right away so future reruns are safe ----
np.save(CKPT_YTRUE, np.asarray(y_true))
np.save(CKPT_YPROB, np.asarray(y_prob))
np.save(CKPT_CLASSNAMES, np.asarray(class_names, dtype=object))

n_classes = len(class_names)
y_true_arr = np.asarray(y_true)
y_prob_arr = np.asarray(y_prob)

THRESHOLDS = np.linspace(0.001, 0.999, 499)   # finer grid for smooth curves

results = {}          # per-class curves
summary_records = []  # per-class summary row

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.ravel()

for i, cls in enumerate(class_names):
    y_bin = (y_true_arr == i).astype(int)
    scores = y_prob_arr[:, i]
    n_pos = int(y_bin.sum())

    fpr, tpr, _ = roc_curve(y_bin, scores)
    roc_auc = auc(fpr, tpr)

    sens_list, spec_list, acc_list = [], [], []
    for t in THRESHOLDS:
        y_hat = (scores >= t).astype(int)
        tp = np.sum((y_hat == 1) & (y_bin == 1))
        tn = np.sum((y_hat == 0) & (y_bin == 0))
        fp = np.sum((y_hat == 1) & (y_bin == 0))
        fn = np.sum((y_hat == 0) & (y_bin == 1))

        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        acc = (tp + tn) / len(y_bin)

        sens_list.append(sens)
        spec_list.append(spec)
        acc_list.append(acc)

    sens_arr = np.array(sens_list)
    spec_arr = np.array(spec_list)
    acc_arr = np.array(acc_list)
    diff_arr = sens_arr - spec_arr

    # accuracy-optimal threshold
    t_acc_idx = np.argmax(acc_arr)
    t_acc = THRESHOLDS[t_acc_idx]

    # sensitivity == specificity: interpolate the true zero-crossing of diff_arr
    sign_change = np.where(np.diff(np.sign(diff_arr)) != 0)[0]
    if len(sign_change) > 0:
        j = sign_change[np.argmin(np.abs(THRESHOLDS[sign_change] - 0.5))]
        x0, x1 = THRESHOLDS[j], THRESHOLDS[j + 1]
        y0, y1 = diff_arr[j], diff_arr[j + 1]
        t_bal = x0 - y0 * (x1 - x0) / (y1 - y0) if (y1 - y0) != 0 else x0
        t_bal = float(np.clip(t_bal, THRESHOLDS.min(), THRESHOLDS.max()))
        t_bal_idx = int(np.argmin(np.abs(THRESHOLDS - t_bal)))
    else:
        t_bal_idx = int(np.argmin(np.abs(diff_arr)))
        t_bal = THRESHOLDS[t_bal_idx]

    # default t=0.50 (closest grid point)
    t50_idx = int(np.argmin(np.abs(THRESHOLDS - 0.50)))

    results[cls] = dict(
        thresholds=THRESHOLDS, sens=sens_arr, spec=spec_arr, acc=acc_arr,
        auc=roc_auc, t_acc=t_acc, t_bal=t_bal, n_pos=n_pos
    )

    summary_records.append({
        "Class": cls,
        "AUC": roc_auc,
        "n_pos": n_pos,
        "Sens_at_0.50": sens_arr[t50_idx],
        "Spec_at_0.50": spec_arr[t50_idx],
        "SensMinusSpec_at_0.50": sens_arr[t50_idx] - spec_arr[t50_idx],
        "t_acc": t_acc,
        "Sens_at_t_acc": sens_arr[t_acc_idx],
        "Spec_at_t_acc": spec_arr[t_acc_idx],
        "Accuracy_at_t_acc": acc_arr[t_acc_idx],
        "t_bal": t_bal,
        "Sens_at_t_bal": sens_arr[t_bal_idx],
        "Spec_at_t_bal": spec_arr[t_bal_idx],
    })

    # ---- plot (shared-legend, shaded-region style) ----
    ax = axes[i]

    ax.fill_between(THRESHOLDS, sens_arr, spec_arr, where=(sens_arr >= spec_arr),
                     color="firebrick", alpha=0.12, interpolate=True)
    ax.fill_between(THRESHOLDS, sens_arr, spec_arr, where=(sens_arr < spec_arr),
                     color="steelblue", alpha=0.12, interpolate=True)

    l_sens, = ax.plot(THRESHOLDS, sens_arr, color="firebrick", lw=1.8, label="Sensitivity")
    l_spec, = ax.plot(THRESHOLDS, spec_arr, color="steelblue", lw=1.8, label="Specificity")
    l_acc, = ax.plot(THRESHOLDS, acc_arr, color="mediumpurple", lw=1.3, linestyle="--", label="Accuracy")

    l_t50 = ax.axvline(0.50, color="gray", linestyle=":", lw=1.2, label="t=0.50 (default)")
    l_tacc = ax.axvline(t_acc, color="black", linestyle="-", lw=1.0,
                         label=r"$t_{acc}$ (accuracy-optimal)")
    l_tbal = ax.axvline(t_bal, color="dimgray", linestyle="--", lw=1.0,
                         label=r"$t_{bal}$ (sens = spec)")

    # circle markers on each curve at t_acc
    ax.plot(t_acc, sens_arr[t_acc_idx], "o", color="firebrick", ms=6, zorder=5)
    ax.plot(t_acc, spec_arr[t_acc_idx], "o", color="steelblue", ms=6, zorder=5)
    # square marker at the true sens=spec crossover
    ax.plot(t_bal, sens_arr[t_bal_idx], "s", color="white", mec="black", mew=1.3, ms=6, zorder=6)

    diff_050 = sens_arr[t50_idx] - spec_arr[t50_idx]
    diff_tacc = sens_arr[t_acc_idx] - spec_arr[t_acc_idx]
    diff_tbal = sens_arr[t_bal_idx] - spec_arr[t_bal_idx]
    textbox = (
        f"t=0.50 (default):      sens - spec = {diff_050:+.3f}\n"
        f"t_acc={t_acc:.2f} (acc.-opt):   sens - spec = {diff_tacc:+.3f}\n"
        f"t_bal={t_bal:.2f} (balanced):   sens - spec = {diff_tbal:+.3f}"
    )
    ax.text(0.02, 0.03, textbox, transform=ax.transAxes, fontsize=8,
            va="bottom", ha="left", family="monospace",
            bbox=dict(boxstyle="round", facecolor="white", edgecolor="gray", alpha=0.9))

    ax.set_title(f"{cls}  (AUC = {roc_auc:.3f},  n = {n_pos})", fontsize=12)
    ax.set_xlabel("Decision threshold t")
    ax.set_ylabel("Operating-point metric")
    ax.set_xlim(0, 1)
    y_min = min(sens_arr.min(), spec_arr.min(), acc_arr.min())
    ax.set_ylim(max(0, y_min - 0.03), 1.01)
    ax.grid(alpha=0.15)

# ---- single shared legend at the top of the figure ----
handles = [l_sens, l_spec, l_acc, l_t50, l_tacc, l_tbal]
labels = [h.get_label() for h in handles]
fig.legend(handles, labels, loc="upper center", ncol=6, fontsize=9,
           bbox_to_anchor=(0.5, 1.03), frameon=False)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(OUTPUT_DIR, "sabbir_threshold_sensitivity_all_classes.png"),
            dpi=600, bbox_inches="tight")
plt.show()

summary_df = pd.DataFrame(summary_records)
summary_df.to_csv(os.path.join(OUTPUT_DIR, "threshold_sensitivity_summary.csv"), index=False)
print(summary_df.to_string(index=False))

# ============================================================
# APPLY CALIBRATED PER-CLASS THRESHOLDS (t_acc) — fixes the
# sensitivity drop at the default 0.50 cutoff, per-class.
#
# NOTE: an earlier "exactly-one-class-passes" rule was a no-op here,
# because Insect Hole's t_acc (0.01) is so low that almost every
# sample clears it -> almost every sample had >1 class "passing" ->
# the fallback (plain argmax) fired on ~100% of samples, so the
# calibrated result was identical to the uncalibrated one.
#
# Correct rule: among the classes that clear THEIR OWN t_acc, pick
# the one with the highest probability. This can override the global
# argmax when the argmax class itself doesn't clear its own bar
# (an overconfident/miscalibrated call) in favor of another class
# that does. Only fall back to plain argmax when NO class clears
# its threshold at all.
# ============================================================
t_acc_per_class = np.array([results[cls]["t_acc"] for cls in class_names])

passed = y_prob_arr >= t_acc_per_class[np.newaxis, :]     # (N, n_classes) bool
n_passed = passed.sum(axis=1)

masked_probs = np.where(passed, y_prob_arr, -np.inf)       # -inf where not passed
y_pred_calibrated = np.argmax(masked_probs, axis=1)

# no class passed -> fall back to plain argmax of raw probabilities
none_passed_mask = n_passed == 0
y_pred_calibrated[none_passed_mask] = np.argmax(y_prob_arr[none_passed_mask], axis=1)

n_overridden = np.sum(y_pred_calibrated != np.argmax(y_prob_arr, axis=1))
print(f"Predictions changed vs plain argmax: {n_overridden} / {len(y_true_arr)}")

from sklearn.metrics import classification_report, accuracy_score

calibrated_acc = accuracy_score(y_true_arr, y_pred_calibrated)
default_acc = accuracy_score(y_true_arr, np.argmax(y_prob_arr, axis=1))

print(f"\nDefault argmax accuracy      : {default_acc:.4f}")
print(f"Calibrated-threshold accuracy: {calibrated_acc:.4f}")
print("\nCalibrated-threshold classification report:")
calibrated_report = classification_report(y_true_arr, y_pred_calibrated, target_names=class_names, digits=4)
print(calibrated_report)

with open(os.path.join(OUTPUT_DIR, "calibrated_threshold_classification_report.txt"), "w") as f:
    f.write(f"Per-class t_acc thresholds: {dict(zip(class_names, t_acc_per_class))}\n\n")
    f.write(calibrated_report)

print("\nSaved Files:")
print("threshold_sensitivity_all_classes.png")
print("threshold_sensitivity_summary.csv")
print("calibrated_threshold_classification_report.txt")